In [12]:


from abc import ABC, abstractmethod
from typing import Any
import re

class BaseExtractor(ABC):
    """Base class for all parameter extractors."""

    @abstractmethod
    def extract(self, text: str) -> Any | None:
        """Extract specific parameter from text."""
        pass
    
class SlugExtractor(BaseExtractor):
    """
    Extracts a slug (12-15 digit number starting with 1) from text using regex only.
    """

    # Regex to find potential slugs: 12-15 digits starting with '1'
    SLUG_PATTERN = re.compile(r"\b1[0-9]{11,14}\b")

    def extract(self, text: str) -> str | None:
        """
        Extract a slug (12-15 digit number starting with 1) from the given text.

        Args:
            text (str): Input text to extract slug from

        Returns:
            Optional[str]: Extracted slug as string, or None if no valid slug found
        """
        if not text:
            return None

        matches = list(self.SLUG_PATTERN.finditer(text))
        if not matches:
            return None

        # Filter valid slugs based on context
        valid_slugs: list[str] = []
        for match in matches:
            if self._is_valid_slug_context(text, match.start(), match.end()):
                valid_slugs.append(match.group())

        if not valid_slugs:
            return None

        # Select the best slug
        return self._select_best_slug(valid_slugs)

    def _is_valid_slug_context(self, text: str, match_start: int, match_end: int, window: int = 20) -> bool:
        """
        Check if the slug is valid by examining its context.
        Specifically filters out slugs that appear with dashes around them (e.g., -1XXXXXXXXXXXX-).
        """
        context_start = max(0, match_start - window)
        context_end = min(len(text), match_end + window)
        context = text[context_start:context_end].lower()
        context = context.replace("\n", "").replace(" ", "")

        # Invalid pattern: slug with dashes around it
        pattern = r"-1[0-9]{11,14}-"
        return not re.search(pattern, context)

    def _select_best_slug(self, slug_candidates: list[str]) -> str | None:
        """
        Select the best slug from candidates:
        - Clean slugs by removing non-digit characters
        - If all identical, return the first
        - Otherwise, return the last (bottom-most) slug
        """
        cleaned_slugs = [re.sub(r"\D", "", slug) for slug in slug_candidates]

        if not cleaned_slugs:
            return None

        if len(set(cleaned_slugs)) == 1:
            return cleaned_slugs[0]
        else:
            return cleaned_slugs[-1]


In [13]:
import pandas as pd
data = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv")

In [14]:
data.document_type.value_counts()

document_type
ladung_va                                 2030
mail_attachments                           984
monierung_mb                               545
attachment_and_transfer_order              535
approved_seizure                           523
court_inbox                                346
fp_protocol                                220
fp_invoice                                 106
vermögensverzeichnis                        75
drittauskunft                               71
approved_attachment_and_transfer_order      64
enforcement_order                           23
tbd                                         22
bailiff_ip                                  20
va_dritt_invoice_protokol_combination       18
contradiction                                3
neg_drittauskunft_hard                       1
Name: count, dtype: int64

In [15]:
dritt = data[data.document_type == "drittauskunft"]
va = data[data.document_type == "vermögensverzeichnis"]
dritt_and_va = data[data.document_type == "va_dritt_invoice_protokol_combination"]

In [16]:
dritt

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va
1344,c518854d-2f65-53af-a4b3-d8b62df46502,e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9,"Gabriele Kruse\nKönigstraße 11, Zi EG 005\nObe...",11.12.2024/ocr-v2_Dokumente_45924_10122024_112...,drittauskunft,"gabriele kruse\nkönigstraße 11, zi eg 005\nobe...",NaN,False,False,NaN,292d99228bc98d5911d52d718ad555c152b3833d968164...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False
5223,ca025357-d3f5-5b97-a780-a4725e3d6239,6cd033bf-c19e-53d4-af67-c5157c19991e,Svea Rietschek\nVolksparkstraße 52\nObergerich...,NaN,drittauskunft,svea rietschek\nvolksparkstraße 52\nobergerich...,NaN,False,False,NaN,6c3cea7cd3eb08d8d6f92e35bb0e258a9f3f35d271df94...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False
5225,829321fd-6887-5d66-ab91-eedbf535ca14,b891083a-0024-5b8e-8e74-3caa640f800b,Gerichtsvollzieher P. Harig\nAmtsgericht Salzg...,NaN,drittauskunft,gerichtsvollzieher p. harig\namtsgericht salzg...,NaN,False,False,NaN,14ad5838a1ecaae3b006a095d34f7d678311ab50cb4abb...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False
5254,8758f08b-eb4f-5d02-b9c4-6d06e6208f7b,1ada309d-2f0a-5f7c-b184-b356624cc518,Obergerichtsvollzieherin G. Samuels\nHESSEN\nb...,NaN,drittauskunft,obergerichtsvollzieherin g. samuels\nhessen\nb...,NaN,False,False,NaN,30b2fe3d8d382e7e79fd5515b6245e68cbb9e86821bd7d...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False
5288,f5dfd4ef-5165-5a8e-96b2-e91be357b220,51643880,Gerichtsvollzieherin beim Amtsgericht Waibling...,NaN,drittauskunft,gerichtsvollzieherin beim amtsgericht waibling...,NaN,False,False,NaN,40c743dda99e173c9fbe465f029efa037f3f22b3cf344d...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5580,4f6c9081-0928-527c-9356-943ba2237293,e3a0150a-cdb8-5a9c-be4a-0f5603361356,R. Müller\nObergerichtsvollzieher\nLerchenweg...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,r. müller\nobergerichtsvollzieher\nlerchenweg ...,NaN,False,False,NaN,fe02c9970e549b20806d238d71c0adfe8a6f3dccdf820a...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nR. Müller\nObergerichtsvollzieher\nL...,False,False
5581,50143302-3ac3-587b-ae25-047d34d94750,99d61bd2-665c-59b1-a24d-264f41c2f474,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,bundeszentralamt\nfür steuern\npostanschrift\n...,NaN,False,False,NaN,93e56ec1cfb7719bf463bf44585e833cdd56fb9689d4f3...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nBundeszentralamt\nfür Steuern\nPOSTA...,False,False
5582,de97e37c-ebf7-5c0e-92be-209db058d9ae,60913326-3481-573b-a96e-72bb2ca9a9ef,Anne-Katrin Götz\nAmtsgericht Dortmund\nOberg...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,anne-katrin götz\namtsgericht dortmund\noberge...,NaN,False,False,NaN,870b8fd6e0da13bafd14ed6e5cec7e79af8c7ae43cae66...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nAnne-Katrin Götz\nAmtsgericht Dortmu...,False,False
5583,c72625b3-d66a-5df5-a933-11f57215838a,2a6e990a-3458-5efe-a8d9-f8b988af1740,Obergerichtsvollzieherin G. Samuels\nHESSEN\n...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,obergerichtsvollzieherin g. samuels\nhessen\nb...,NaN,False,False,NaN,6eb9335329883f671aa167608aa4108d45a5e0fcbbbb86...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nObergerichtsvollzieherin G. Samuels\...,False,False


In [18]:
slug_extractor = SlugExtractor()
dritt['extracted_slug'] = dritt['cleaned_text'].apply(slug_extractor.extract)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_42440/4021042366.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dritt['extracted_slug'] = dritt['cleaned_text'].apply(slug_extractor.extract)


In [19]:
dritt

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va,extracted_slug
1344,c518854d-2f65-53af-a4b3-d8b62df46502,e4b3c249-4de5-5cee-8e8f-32bd8ad0b5d9,"Gabriele Kruse\nKönigstraße 11, Zi EG 005\nObe...",11.12.2024/ocr-v2_Dokumente_45924_10122024_112...,drittauskunft,"gabriele kruse\nkönigstraße 11, zi eg 005\nobe...",NaN,False,False,NaN,292d99228bc98d5911d52d718ad555c152b3833d968164...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False,140069758252
5223,ca025357-d3f5-5b97-a780-a4725e3d6239,6cd033bf-c19e-53d4-af67-c5157c19991e,Svea Rietschek\nVolksparkstraße 52\nObergerich...,NaN,drittauskunft,svea rietschek\nvolksparkstraße 52\nobergerich...,NaN,False,False,NaN,6c3cea7cd3eb08d8d6f92e35bb0e258a9f3f35d271df94...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False,180878666809
5225,829321fd-6887-5d66-ab91-eedbf535ca14,b891083a-0024-5b8e-8e74-3caa640f800b,Gerichtsvollzieher P. Harig\nAmtsgericht Salzg...,NaN,drittauskunft,gerichtsvollzieher p. harig\namtsgericht salzg...,NaN,False,False,NaN,14ad5838a1ecaae3b006a095d34f7d678311ab50cb4abb...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False,182984186458
5254,8758f08b-eb4f-5d02-b9c4-6d06e6208f7b,1ada309d-2f0a-5f7c-b184-b356624cc518,Obergerichtsvollzieherin G. Samuels\nHESSEN\nb...,NaN,drittauskunft,obergerichtsvollzieherin g. samuels\nhessen\nb...,NaN,False,False,NaN,30b2fe3d8d382e7e79fd5515b6245e68cbb9e86821bd7d...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False,163836166096
5288,f5dfd4ef-5165-5a8e-96b2-e91be357b220,51643880,Gerichtsvollzieherin beim Amtsgericht Waibling...,NaN,drittauskunft,gerichtsvollzieherin beim amtsgericht waibling...,NaN,False,False,NaN,40c743dda99e173c9fbe465f029efa037f3f22b3cf344d...,s3://pair-data-engineering-new/ocr_prepared_ou...,False,NaN,NaN,False,142333753093
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5580,4f6c9081-0928-527c-9356-943ba2237293,e3a0150a-cdb8-5a9c-be4a-0f5603361356,R. Müller\nObergerichtsvollzieher\nLerchenweg...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,r. müller\nobergerichtsvollzieher\nlerchenweg ...,NaN,False,False,NaN,fe02c9970e549b20806d238d71c0adfe8a6f3dccdf820a...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nR. Müller\nObergerichtsvollzieher\nL...,False,False,163495927085
5581,50143302-3ac3-587b-ae25-047d34d94750,99d61bd2-665c-59b1-a24d-264f41c2f474,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,bundeszentralamt\nfür steuern\npostanschrift\n...,NaN,False,False,NaN,93e56ec1cfb7719bf463bf44585e833cdd56fb9689d4f3...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nBundeszentralamt\nfür Steuern\nPOSTA...,False,False,120173044047
5582,de97e37c-ebf7-5c0e-92be-209db058d9ae,60913326-3481-573b-a96e-72bb2ca9a9ef,Anne-Katrin Götz\nAmtsgericht Dortmund\nOberg...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,anne-katrin götz\namtsgericht dortmund\noberge...,NaN,False,False,NaN,870b8fd6e0da13bafd14ed6e5cec7e79af8c7ae43cae66...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nAnne-Katrin Götz\nAmtsgericht Dortmu...,False,False,168097557412
5583,c72625b3-d66a-5df5-a933-11f57215838a,2a6e990a-3458-5efe-a8d9-f8b988af1740,Obergerichtsvollzieherin G. Samuels\nHESSEN\n...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,obergerichtsvollzieherin g. samuels\nhessen\nb...,NaN,False,False,NaN,6eb9335329883f671aa167608aa4108d45a5e0fcbbbb86...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nObergerichtsvollzieherin G. Samuels\...,False,False,163836166096


In [23]:
lenths = dritt['extracted_slug'].dropna().apply(len)
lenths.value_counts()

extracted_slug
12    69
Name: count, dtype: int64

In [25]:
dritt_no_slug = dritt[dritt['extracted_slug'].isna()]
dritt_no_slug

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va,extracted_slug
5538,ba9b1fcd-81a8-51d4-a593-7f9b043236a2,acd659dc-1c2e-59fb-b29e-56cd3e95f4eb,Gerichtsvollzieherin\nWaldhofer Straße 17\nKa...,data/aftercourt/drittauskunf_with_invoice/1_dr...,drittauskunft,gerichtsvollzieherin\nwaldhofer straße 17\nkat...,NaN,False,False,NaN,bd08800d05ff7cf06ea333a5853b3ee7fd193fa516eb1f...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nGerichtsvollzieherin\nWaldhofer Stra...,True,False,None
5566,43d92153-88ce-58b5-96cc-67b1f4c902dc,55d76158-1190-5c2c-b58a-fa2252117f16,Bundeszentralamt\nfür Steuern\nPOSTANSCHRIFT\...,data/aftercourt/drittauskunft_without_invoice/...,drittauskunft,bundeszentralamt\nfür steuern\npostanschrift\n...,NaN,False,False,NaN,d2a52b5ed580d39427974c22e0db6e655a59b07917ff12...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,<page_1>\nBundeszentralamt\nfür Steuern\nPOSTA...,False,False,None


In [28]:
print(dritt_no_slug['cleaned_text'].tolist()[1])

bundeszentralamt
für steuern
postanschrift
bundeszentralamt für steuern, 11055 berlin
hausanschrift
dgz-ring 12, 13086 berlin
bearbeitet von
gerichtsvollzieher rüngeling
cardenap 7
steuerabteilung national
kontenabrufverfahren
38518 gifhorn
+49 (0) 2 28 40 6-3600
+49 (0) 2 28 40 6-4408
e-mail
kontenabruf@bzst.bund.de
internet
www.bzst.bund.de
betreff
kontenabrufersuchen nach §§ 93, 93b abgabenordnung (ao)
bezug
ihr kontenabrufersuchen vom 02.01.2026 az: dr il 8/26
anlagen
st ii 4 s 0229a kevizz 6194/26
(bei antwort bitte angeben)
datum
05.01.2026
ihr oben genanntes kontenabrufersuchen ist im rahmen des automatisierten abrufs von
kontoinformationen bearbeitet worden. die ergebnisse dieses abrufs entnehmen sie bitte der
nachfolgenden anlage.
bitte beachten sie
dass die verantwortung für die zulässigkeit des datenabrufs und die datenübermittlung die
ersuchende behörde trägt (§ 93b abs. 3 ao),
die informations- und dokumentationspflichten gemäß § 93 abs. 9 und 10 ao sowie den
anwendungserl

In [32]:
va['extracted_slug'] = va['cleaned_text'].apply(slug_extractor.extract)
va['extracted_slug'].apply(lambda x: len(x) if x else None).value_counts()

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_42440/1003464000.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  va['extracted_slug'] = va['cleaned_text'].apply(slug_extractor.extract)


extracted_slug
12.0    62
Name: count, dtype: int64

In [34]:
va_no_slug = va[va['extracted_slug'].isna()]
va_no_slug

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,text_w_pages,is_da_with_invoice,is_va,extracted_slug
3751,0cf7b4a3-9112-5b90-b6be-481f4f846992,139dfe4e-9ee4-5561-9e26-ce256bf482fa,Anlage zur Niederschrift d.:\nGV Dominik Schna...,29.07.2024/ocr-v2_VV_46224_140674123116.pdf,vermögensverzeichnis,anlage zur niederschrift d.:\ngv dominik schna...,NaN,False,False,NaN,4696d06f240759c015063ccf50b858d6556aa52bba3773...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5287,73f21aef-cb62-5165-8384-b2859a5beb5b,51428247,Anlage zur Niederschrift d.:\nGV J. Korn\nvom:...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\ngv j. korn\nvom:...,NaN,False,False,NaN,7d526c456e0e6f7ba25db0a35ec7e496d78289d92a6071...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5293,249d8989-7afe-5f15-b020-57838e595fe3,51774647,Anlage zur Niederschrift d.:\nOGV Theo Schmitz...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\nogv theo schmitz...,NaN,False,False,NaN,c8d0c898eda460364a345c4ee654ea53104924be414ada...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5300,3f306157-6c99-57c5-bcb3-9df4b58ae2d6,50219916,Anlage zur Niederschrift d.:\nGV Nico Stolzen\...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\ngv nico stolzen\...,NaN,False,False,NaN,c3a5f5b3c3333f6c09cfb385cc0b39d7e0aef33147ebf8...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5306,d0f0e78b-5994-5c26-b912-cbf628e92b12,51771792,Anlage zur Niederschrift d.:\nOGV Doreen Trieb...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\nogv doreen trieb...,NaN,False,False,NaN,0a5f2fde998db2656f636931dd937eec56045a1524f364...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5323,3bc46a02-531e-5ff6-8ff7-960179af5c95,50450147,Anlage zur Niederschrift d.:\nOGV Torsten Olbr...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\nogv torsten olbr...,NaN,False,False,NaN,6e629d0ec3f95a571c49193e693ba531ba7952eedfdc86...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5329,2f575e42-e662-53f4-a558-cfb690008ee0,51174964,Anlage zur Niederschrift d.:\nOGV Sascha Deter...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\nogv sascha deter...,NaN,False,False,NaN,6192616c01dd892e865fb330d3ea25dce67ee4004f7e24...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5339,b44a9bcb-2ab0-5162-af29-dbfae93edfa9,50396500,Anlage zur Niederschrift d.:\nOGVin S. Jungman...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\nogvin s. jungman...,NaN,False,False,NaN,1c3f275fa209ee2345eb781e7084ecbe800f0d8ee18d39...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5342,2fadc54c-76cf-5235-8203-d5c73e8e629a,51881467,Anlage zur Niederschrift d.:\nGVin Nicole Schr...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\ngvin nicole schr...,NaN,False,False,NaN,091294a91559890d71bbb7cc745a92bc9f95437aa0fe9d...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None
5343,3e5e56bd-9384-57c0-a364-d9feef5ac9b0,50910730,Anlage zur Niederschrift d.:\nOGV Otto Fuß\nvo...,NaN,vermögensverzeichnis,anlage zur niederschrift d.:\nogv otto fuß\nvo...,NaN,False,False,NaN,a62fa83b350ba5f0d56d1cae7f7f580d2a0aaff5e77dcc...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,NaN,NaN,True,None


In [38]:
import os
import sys
sys.path.append('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/')

In [40]:
from utils.prod_utils import get_data_by_attachment_id

from python_utilities.db_connection import DbConnection
import boto3
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
# Create session with specific profile
session = boto3.Session(profile_name='739275445236_DataScienceUser')
s3 = session.client('s3')

for a_id in va_no_slug['attachment_id'].tolist():
    get_data_by_attachment_id(a_id, analytics_db, s3, pdf_download=True, pdf_download_dir="/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp")


INFO [2026-06-19 13:24:15] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO [2026-06-19 13:24:16] - Found credentials in shared credentials file: ~/.aws/credentials


No data found for attachment_id: 139dfe4e-9ee4-5561-9e26-ce256bf482fa

📎 ATTACHMENT SUMMARY
Attachment ID:   51428247
File Name:       51428247_VV_124325_4892.pdf

│ 📄 Document S3:
│    s3://pair-data-engineering-new/ocr_source_files/2026-02-01/egvp_id_300574/51428247_VV_124325_4892.pdf
│
│ 📝 Textract Output:
│    s3://pair-data-engineering-new/ocr_prepared_output/1c49445c91e65ab3fc89af3ea190c0c932d648bd99ded9338ec9eb9f88f613a4.json
│
│ 📊 Text Length:  11355 characters
│
│ 📥 PDF Downloaded: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/51428247.pdf
│
│ 🤖 PREDICTIONS:
│    ⚠️  NOT LADUNG OR PFUB! 
│    Prob Ladung:              '0.86'
│    Prob Pfub:                '0.0'
│    Prob Vermogenverzeichnis: N/A
│    — invoice_detection_egvp —
│    Start Page:        N/A
│    End Page:          N/A
│    Is Invoice Inside: N/A
│    — pfub_erlass_egvp —
│    Is Pfub:           N/A
│    Is Invoice Inside: N/A
└─────────────────────────────────────────────────────────